In [0]:
-- SHOW CONNECTIONS;

create  CONNECTION IF NOT EXISTS youtube_earthqueak_conn
type HTTP
OPTIONS(
   host  = 'https://earthquake.usgs.gov',
   port = 443,
   base_path = '/earthquakes/feed/v1.0/',
   bearer_token = 'na'

)



In [0]:
import requests
import json

url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"
response = requests.get(url)
data = json.loads(response.text)
-- data
-- CREATE




In [0]:
%python

import requests
import json
import pandas as pd
from pyspark.sql import functions as F

# 1. 拼接视频中的完整 API URL
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
print(f"开始请求数据源: {url}")

# 2. 发起 HTTP GET 请求
response = requests.get(url)

if response.status_code == 200:
    print("API 请求成功！开始解析 JSON...")
    
    # 3. 将 JSON 文本解析为 Python 字典
    raw_data = response.json()
    
    features_list = raw_data.get('features', [])
    
    # 4. 把复杂的 JSON 列表转换为扁平化的结构
    flattened_data = []
    for feature in features_list:
        properties = feature.get('properties', {})
        geometry = feature.get('geometry', {})
        combined_record = {**properties, **geometry}
        flattened_data.append(combined_record)

    # 5. 使用 Pandas 作为桥梁，转换为 Spark DataFrame
    pdf = pd.DataFrame(flattened_data)
    spark_df = spark.createDataFrame(pdf)
    
    print("数据成功转换为 Spark DataFrame！")
    
    # 6. 预览数据
    display(spark_df)
    
    # 7. 【关键步骤】：把数据写入你自己的 catalog 和 schema 下，作为 Bronze 表！
    # 假设你在视频里建的 catalog 叫 hive_metastore，schema 叫 default
    # 如果视频里教你建了 Unity Catalog，比如叫 my_catalog.my_schema，请把下面字符串替换掉
    spark_df.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.default.earthquake_bronze")
    print("数据已成功保存为 Bronze 表！可以继续跟着视频做后续了！")

else:
    print(f"API 请求失败，状态码: {response.status_code}")